# Clipper ASR-Worker: Qwen3-ASR auf Colab (T4)

Dieses Notebook macht **nur Spracherkennung**. Alles andere (Clip-Länge, Auswahl, Schnitt, Untertitel) steuert dein lokaler Rechner mit `python -m clipper run VIDEO`. Es wird kein Code geklont und kein Video hochgeladen: Der Orchestrator schickt pro Auftrag nur die Tonspur (Opus, ca. 22 MB pro Stunde) und bekommt Wörter mit Zeitstempeln zurück.

- **Silero VAD** findet Sprache und schneidet sie in ≤ 30-s-Stücke an Pausen (kein Wort wird geteilt).
- **Qwen3-ASR-1.7B** transkribiert (Sprache vorgegeben oder automatisch erkannt).
- **Qwen3-ForcedAligner-0.6B** (de/en/fr/es/it/pt/ru) bzw. **MMS-300m-CTC-Aligner** (Arabisch und Rest) liefert die Wort-Zeitstempel.
- Alle Modelle bleiben zwischen den Aufträgen im GPU-Speicher.

## Vor dem Start
1. *Laufzeit → Laufzeittyp ändern* → **T4 GPU**.
2. Einmalig: links das **Schlüssel-Symbol** → Secret `CLIPPER_TOKEN` anlegen (gleicher Wert wie in Kaggle und in `clipper/.env`), *Notebook-Zugriff* einschalten.
3. *Laufzeit → Alle ausführen*. Die `export`-Zeilen aus Abschnitt 5 in `clipper/.env` eintragen.
4. Abschnitt 6 laufen lassen (hält die Sitzung wach). Link und Token nicht teilen.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Keine GPU: Laufzeit > Laufzeittyp ändern > T4 GPU'
print(torch.cuda.get_device_name(0))

## 1. Pakete installieren

In [ ]:
!pip -q install "qwen-asr>=0.0.6" "silero-vad>=5.1" "transformers>=4.51.0" accelerate fastapi uvicorn
!pip -q install "git+https://github.com/MahmoudAshraf97/ctc-forced-aligner.git"
!python -c "import qwen_asr, silero_vad, ctc_forced_aligner, fastapi, uvicorn; print('ok')"

## 2. ASR-Engine
VAD → Chunks → Qwen3-ASR → Aligner → Wörter mit Zeitstempeln. Das Ausgabeformat entspricht `clipper/models.py` (`Transcript`).

In [ ]:
import difflib, gc, re, subprocess, unicodedata
from collections import Counter
from dataclasses import asdict, dataclass

import numpy as np
import torch

SR = 16000
ASR_MODEL = 'Qwen/Qwen3-ASR-1.7B'
QWEN_ALIGNER = 'Qwen/Qwen3-ForcedAligner-0.6B'
CTC_ALIGNER = 'MahmoudAshraf/mms-300m-1130-forced-aligner'
BATCH = 8
DEVICE = 'cuda:0'
DTYPE = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16

# code -> (Qwen3-ASR language name, ISO 639-3 for the MMS aligner)
LANGUAGES = {
    'ar': ('Arabic', 'ara'), 'de': ('German', 'deu'), 'en': ('English', 'eng'), 'fr': ('French', 'fra'),
    'es': ('Spanish', 'spa'), 'it': ('Italian', 'ita'), 'pt': ('Portuguese', 'por'), 'ru': ('Russian', 'rus'),
    'tr': ('Turkish', 'tur'), 'fa': ('Persian', 'fas'), 'hi': ('Hindi', 'hin'), 'nl': ('Dutch', 'nld'),
    'pl': ('Polish', 'pol'), 'id': ('Indonesian', 'ind'),
}
QWEN_ALIGNER_LANGS = {'zh', 'en', 'yue', 'fr', 'de', 'it', 'ja', 'ko', 'pt', 'ru', 'es'}
_NAME_TO_CODE = {name.lower(): code for code, (name, _) in LANGUAGES.items()}
_NAME_TO_CODE.update({'chinese': 'zh', 'cantonese': 'yue', 'japanese': 'ja', 'korean': 'ko'})


def language_code(value):
    if not value:
        return None
    value = value.strip().lower()
    if value in LANGUAGES or value in QWEN_ALIGNER_LANGS:
        return value
    return _NAME_TO_CODE.get(value)


@dataclass
class Word:
    text: str
    start: float
    end: float
    aligned: bool = True


@dataclass
class Chunk:
    start: float
    end: float
    text: str = ''
    language: str | None = None


# ---------------------------------------------------------------- audio

def load_audio(path):
    proc = subprocess.run(['ffmpeg', '-nostdin', '-v', 'error', '-i', path, '-vn', '-ac', '1', '-ar', str(SR),
                           '-f', 'f32le', 'pipe:1'], capture_output=True)
    if proc.returncode != 0:
        raise RuntimeError('Audio decode failed: ' + proc.stderr.decode(errors='replace')[-500:])
    return np.frombuffer(proc.stdout, dtype=np.float32).copy()


def rms_envelope(audio, hop=0.01):
    step = int(SR * hop)
    n = len(audio) // step
    if n == 0:
        return np.full(1, -100.0, dtype=np.float32)
    frames = audio[: n * step].reshape(n, step).astype(np.float64)
    return (20 * np.log10(np.sqrt(np.mean(frames ** 2, axis=1) + 1e-12) + 1e-9)).astype(np.float32)


# ---------------------------------------------------------------- text <-> aligner tokens (as clipper/textalign.py)

_ARABIC_DIACRITICS = re.compile(r'[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED\u0640]')


def normalize(token):
    token = unicodedata.normalize('NFKC', token).lower()
    token = _ARABIC_DIACRITICS.sub('', token)
    token = token.replace('أ', 'ا').replace('إ', 'ا').replace('آ', 'ا').replace('ى', 'ي').replace('ة', 'ه')
    return ''.join(ch for ch in token if unicodedata.category(ch)[0] in 'LN')


def tokenize(text):
    tokens = []
    for raw in text.split():
        if not normalize(raw) and tokens:
            tokens[-1] += raw
        else:
            tokens.append(raw)
    if tokens and not normalize(tokens[0]):
        tokens = tokens[1:]
    return tokens


def _interpolate_missing(words, span):
    i, n = 0, len(words)
    while i < n:
        if words[i].aligned:
            i += 1
            continue
        j = i
        while j < n and not words[j].aligned:
            j += 1
        left = words[i - 1].end if i > 0 else span[0]
        right = words[j].start if j < n else span[1]
        right = max(right, left)
        weights = [max(1, len(normalize(w.text))) for w in words[i:j]]
        total, cursor = float(sum(weights)), left
        for w, wt in zip(words[i:j], weights):
            step = (right - left) * wt / total
            w.start, w.end = cursor, cursor + step
            cursor += step
        i = j


def _enforce_monotonic(words, span):
    prev_end = span[0]
    for w in words:
        w.start = max(w.start, prev_end, span[0])
        w.end = min(max(w.end, w.start + 0.02), max(span[1], w.start + 0.02))
        prev_end = w.end


def attach_times(tokens, aligned, span):
    n = len(tokens)
    starts, ends = [None] * n, [None] * n
    norm_tokens = [normalize(t) for t in tokens]
    norm_aligned = [normalize(a[0]) for a in aligned]
    matcher = difflib.SequenceMatcher(None, norm_tokens, norm_aligned, autojunk=False)
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag == 'equal' or (tag == 'replace' and i2 - i1 == j2 - j1):
            for k in range(i2 - i1):
                starts[i1 + k], ends[i1 + k] = aligned[j1 + k][1], aligned[j1 + k][2]
        elif tag == 'replace':
            # Different tokenisation (e.g. aligner split a compound): spread the span by characters.
            t0, t1 = aligned[j1][1], aligned[j2 - 1][2]
            weights = [max(1, len(norm_tokens[i])) for i in range(i1, i2)]
            total, cursor = float(sum(weights)), t0
            for i, w in zip(range(i1, i2), weights):
                step = (t1 - t0) * w / total
                starts[i], ends[i] = cursor, cursor + step
                cursor += step
    words = [Word(t, s if s is not None else -1.0, e if e is not None else -1.0, s is not None)
             for t, s, e in zip(tokens, starts, ends)]
    _interpolate_missing(words, span)
    _enforce_monotonic(words, span)
    return words


def interpolate_words(text, speech, span):
    tokens = tokenize(text)
    regions = [(max(a, span[0]), min(b, span[1])) for a, b in speech if b > span[0] and a < span[1]] or [span]
    total_speech = sum(b - a for a, b in regions) or 1e-6
    weights = [max(1, len(normalize(t))) for t in tokens]
    total_w = float(sum(weights)) or 1.0

    def at(fraction):
        remaining = fraction * total_speech
        for a, b in regions:
            if remaining <= b - a:
                return a + remaining
            remaining -= b - a
        return regions[-1][1]

    words, acc = [], 0.0
    for tok, wt in zip(tokens, weights):
        start = at(acc / total_w)
        acc += wt
        words.append(Word(tok, start, at(acc / total_w), aligned=False))
    return words


# ---------------------------------------------------------------- models (loaded once, kept on the GPU)

_MODELS = {}


def get_model(kind):
    if kind not in _MODELS:
        if kind == 'vad':
            from silero_vad import load_silero_vad
            _MODELS[kind] = load_silero_vad()
        elif kind == 'asr':
            from qwen_asr import Qwen3ASRModel
            _MODELS[kind] = Qwen3ASRModel.from_pretrained(ASR_MODEL, dtype=DTYPE, device_map=DEVICE,
                                                          max_inference_batch_size=BATCH, max_new_tokens=1024)
        elif kind == 'qwen_aligner':
            from qwen_asr import Qwen3ForcedAligner
            _MODELS[kind] = Qwen3ForcedAligner.from_pretrained(QWEN_ALIGNER, dtype=DTYPE, device_map=DEVICE)
        elif kind == 'ctc_aligner':
            from ctc_forced_aligner import load_alignment_model
            _MODELS[kind] = load_alignment_model('cuda', CTC_ALIGNER, dtype=torch.float16)
    return _MODELS[kind]


def free_cache():
    gc.collect()
    torch.cuda.empty_cache()


# ---------------------------------------------------------------- VAD + chunking

def detect_speech(audio, threshold):
    from silero_vad import get_speech_timestamps
    stamps = get_speech_timestamps(torch.from_numpy(audio), get_model('vad'), threshold=threshold, sampling_rate=SR,
                                   min_speech_duration_ms=200, min_silence_duration_ms=250, speech_pad_ms=80)
    return [(s['start'] / SR, s['end'] / SR) for s in stamps]


def _split_long(seg, max_sec, env_db, hop):
    # Split a speech region longer than max_sec at its quietest points.
    out, start, end = [], seg[0], seg[1]
    while end - start > max_sec:
        lo, hi = start + max_sec * 0.6, start + max_sec
        i0, i1 = int(lo / hop), max(int(lo / hop) + 1, int(hi / hop))
        window = env_db[i0:i1]
        cut = (i0 + int(np.argmin(window))) * hop if len(window) else hi
        out.append((start, cut))
        start = cut
    out.append((start, end))
    return out


def plan_chunks(speech, env_db, max_sec, hop=0.01, max_gap=3.0):
    pieces = []
    for seg in speech:
        pieces.extend(_split_long(seg, max_sec, env_db, hop))
    chunks = []
    for s, e in pieces:
        if chunks and e - chunks[-1].start <= max_sec and s - chunks[-1].end <= max_gap:
            chunks[-1].end = e
        else:
            chunks.append(Chunk(s, e))
    return chunks


# ---------------------------------------------------------------- ASR + alignment

_REPEAT = re.compile(r'(\b\S+\b)(?:\s+\1\b){4,}')


def _clean_text(text):
    # Guard against ASR loops on music/noise ("la la la la la ...").
    text = _REPEAT.sub(lambda m: f'{m.group(1)} {m.group(1)}', text)
    return re.sub(r'\s+', ' ', text).strip()


def _clip(audio, c):
    return audio[int(c.start * SR): int(c.end * SR)]


def run_asr(audio, chunks, language, context, progress):
    model = get_model('asr')
    forced = LANGUAGES[language][0] if language != 'auto' else None
    for b in range(0, len(chunks), BATCH):
        group = chunks[b: b + BATCH]
        results = model.transcribe(audio=[(_clip(audio, c), SR) for c in group],
                                   language=[forced] * len(group), context=[context] * len(group))
        for chunk, res in zip(group, results):
            chunk.text = _clean_text(res.text or '')
            chunk.language = language if forced else language_code(res.language)
        progress(f'ASR {min(b + BATCH, len(chunks))}/{len(chunks)} Chunks')


def align_qwen(audio, chunks, progress):
    aligner, out = get_model('qwen_aligner'), {}

    def align_group(group):
        results = aligner.align(audio=[(_clip(audio, c), SR) for c in group], text=[c.text for c in group],
                                language=[LANGUAGES[c.language][0] for c in group])
        for chunk, res in zip(group, results):
            out[id(chunk)] = [(it.text, chunk.start + float(it.start_time), chunk.start + float(it.end_time))
                              for it in list(getattr(res, 'items', res))]

    for b in range(0, len(chunks), BATCH):
        group = chunks[b: b + BATCH]
        try:
            align_group(group)
        except Exception as exc:
            # One by one, so a single bad chunk (or OOM on a padded batch) only costs that chunk.
            print('Qwen aligner batch failed, retrying chunk by chunk:', exc)
            for chunk in group:
                try:
                    align_group([chunk])
                except Exception as exc_one:
                    print(f'Qwen aligner failed for {chunk.start:.1f}-{chunk.end:.1f}s ({exc_one}); interpolating')
                    free_cache()
        progress(f'Zeitstempel (Qwen) {min(b + BATCH, len(chunks))}/{len(chunks)} Chunks')
    return out


def align_ctc(audio, chunks, progress):
    from ctc_forced_aligner import generate_emissions, get_alignments, get_spans, postprocess_results, preprocess_text
    model, tokenizer = get_model('ctc_aligner')
    out = {}
    for n, chunk in enumerate(chunks, 1):
        try:
            wav = torch.from_numpy(_clip(audio, chunk)).to(device=model.device, dtype=model.dtype)
            emissions, stride = generate_emissions(model, wav, batch_size=4)
            iso3 = LANGUAGES.get(chunk.language or '', ('', 'eng'))[1]
            # MMS normalisation deletes digit-only words (empty tokens break alignment); they get interpolated.
            text = ' '.join(t for t in tokenize(chunk.text) if not normalize(t).isdigit())
            if not text:
                continue
            tokens_starred, text_starred = preprocess_text(text, romanize=True, language=iso3)
            segments, scores, blank = get_alignments(emissions, tokens_starred, tokenizer)
            spans = get_spans(tokens_starred, segments, blank)
            stamps = postprocess_results(text_starred, spans, stride, scores)
            out[id(chunk)] = [(s['text'], chunk.start + float(s['start']), chunk.start + float(s['end'])) for s in stamps]
        except Exception as exc:
            print(f'CTC alignment failed for {chunk.start:.1f}-{chunk.end:.1f}s ({exc}); interpolating')
            free_cache()
        if n % 10 == 0 or n == len(chunks):
            progress(f'Zeitstempel (MMS) {n}/{len(chunks)} Chunks')
    return out


def transcribe(path, language='auto', context='', vad_threshold=0.5, chunk_max_sec=30.0, progress=print):
    progress('Audio dekodieren')
    audio = load_audio(path)
    if not len(audio):
        raise RuntimeError('Keine Tonspur')
    env_db = rms_envelope(audio)
    progress('Sprache erkennen (VAD)')
    speech = detect_speech(audio, vad_threshold)
    if not speech:
        raise RuntimeError('Keine Sprache erkannt')
    chunks = plan_chunks(speech, env_db, chunk_max_sec)
    progress(f'{len(speech)} Sprachabschnitte -> {len(chunks)} Chunks')

    run_asr(audio, chunks, language, context, progress)
    # Drop empty / punctuation-only outputs ("...", "♪"): nothing to caption, and they break aligners.
    chunks = [c for c in chunks if any(normalize(t) for t in tokenize(c.text))]
    if not chunks:
        raise RuntimeError('ASR lieferte keinen Text')
    langs = Counter(c.language for c in chunks if c.language)
    dominant = language if language != 'auto' else (langs.most_common(1)[0][0] if langs else None)
    for c in chunks:
        if c.language not in LANGUAGES:
            c.language = dominant if dominant in LANGUAGES else 'en'

    aligned = {}
    qwen_chunks = [c for c in chunks if c.language in QWEN_ALIGNER_LANGS]
    ctc_chunks = [c for c in chunks if c.language not in QWEN_ALIGNER_LANGS]
    if qwen_chunks:
        aligned.update(align_qwen(audio, qwen_chunks, progress))
    if ctc_chunks:
        aligned.update(align_ctc(audio, ctc_chunks, progress))

    words = []
    for c in chunks:
        stamps = aligned.get(id(c))
        words.extend(attach_times(tokenize(c.text), stamps, (c.start, c.end)) if stamps
                     else interpolate_words(c.text, speech, (c.start, c.end)))
    misses = sum(not w.aligned for w in words)
    progress(f'Fertig: {len(words)} Wörter, Sprache {dominant}, {misses} interpoliert')
    return {
        'source': str(path),
        'duration': round(len(audio) / SR, 3),
        'language': dominant,
        'words': [asdict(w) for w in words],
        'speech': [(round(a, 3), round(b, 3)) for a, b in speech],
        'chunks': [{'start': round(c.start, 3), 'end': round(c.end, 3), 'language': c.language, 'text': c.text} for c in chunks],
    }


print('ASR-Engine definiert.')

## 3. Modelle laden
Lädt alle Modelle einmal herunter und auf die GPU (zusammen ca. 6 GB VRAM), damit der erste Auftrag nicht wartet.

In [ ]:
for kind in ('vad', 'asr', 'qwen_aligner', 'ctc_aligner'):
    get_model(kind)
    print('geladen:', kind)
print(f'VRAM belegt: {torch.cuda.memory_allocated() / 2**30:.1f} GiB')

## 4. ASR-Server starten
Job-API, weil eine lange Transkription das 100-s-Limit des Cloudflare-Tunnels überschreiten würde:

- `POST /jobs?language=auto&context=...&vad_threshold=0.5&chunk_max_sec=30` mit der Audiodatei als Body → `{"id": ...}`
- `GET /jobs/{id}` → `status` (`queued`/`running`/`done`/`error`), `stage` (Fortschritt), `result` (Transkript)
- `GET /health` (ohne Token), `GET /auth` (prüft den Token)

Alle Werte schickt der Orchestrator pro Auftrag mit; Aufträge laufen nacheinander auf der GPU.

In [ ]:
import queue, secrets, threading, time, traceback, urllib.request, uuid
from pathlib import Path

import uvicorn
from fastapi import FastAPI, Header, HTTPException, Request

API_TOKEN = ''
try:
    from google.colab import userdata
    API_TOKEN = userdata.get('CLIPPER_TOKEN') or ''
except Exception:
    pass
if not API_TOKEN:
    API_TOKEN = secrets.token_urlsafe(32)
    print('Kein Secret CLIPPER_TOKEN gefunden - zufälliger Token für diese Sitzung.')
PORT = 8002
MAX_UPLOAD = 500 * 2**20
UPLOADS = Path('/content/asr_jobs')
UPLOADS.mkdir(exist_ok=True)
JOBS, QUEUE, CURRENT = {}, queue.Queue(), {'id': None}


def worker():
    while True:
        job_id = QUEUE.get()
        job = JOBS[job_id]
        CURRENT['id'] = job_id
        job['status'], job['started'] = 'running', time.time()

        def progress(message, job=job):
            job['stage'] = message

        try:
            job['result'] = transcribe(job['path'], progress=progress, **job['params'])
            job['status'] = 'done'
        except Exception as exc:
            traceback.print_exc()
            job['status'], job['error'] = 'error', f'{type(exc).__name__}: {exc}'
            free_cache()
        finally:
            CURRENT['id'] = None
            Path(job['path']).unlink(missing_ok=True)
            print(f"Job {job_id[:8]}: {job['status']} nach {time.time() - job['started']:.0f} s", flush=True)


def check_token(authorization):
    if not secrets.compare_digest(authorization or '', f'Bearer {API_TOKEN}'):
        raise HTTPException(status_code=401, detail='invalid token')


app = FastAPI()


@app.get('/health')
def health():
    return {'status': 'ok', 'queued': QUEUE.qsize(), 'running': CURRENT['id'] is not None}


@app.get('/auth')
def auth(authorization: str | None = Header(None)):
    check_token(authorization)
    return {'ok': True}


@app.post('/jobs')
async def create_job(request: Request, language: str = 'auto', context: str = '', vad_threshold: float = 0.5,
                     chunk_max_sec: float = 30.0, authorization: str | None = Header(None)):
    check_token(authorization)
    if language != 'auto' and language not in LANGUAGES:
        raise HTTPException(status_code=400, detail=f'unknown language {language}')
    body = await request.body()
    if not body or len(body) > MAX_UPLOAD:
        raise HTTPException(status_code=413 if body else 400, detail='empty or too large upload')
    job_id = uuid.uuid4().hex
    path = UPLOADS / f'{job_id}.audio'
    path.write_bytes(body)
    params = {'language': language, 'context': context[:2000],
              'vad_threshold': min(max(vad_threshold, 0.1), 0.9), 'chunk_max_sec': min(max(chunk_max_sec, 5.0), 30.0)}
    JOBS[job_id] = {'status': 'queued', 'stage': 'wartet', 'path': str(path), 'params': params}
    QUEUE.put(job_id)
    print(f'Job {job_id[:8]}: {len(body) / 1e6:.1f} MB, {params}', flush=True)
    return {'id': job_id}


@app.get('/jobs/{job_id}')
def job_status(job_id: str, authorization: str | None = Header(None)):
    check_token(authorization)
    job = JOBS.get(job_id)
    if job is None:
        raise HTTPException(status_code=404, detail='unknown job')
    return {k: job.get(k) for k in ('status', 'stage', 'error', 'result')}


# Re-running this cell keeps the running server (and its token); restart the runtime to change it.
if not globals().get('_asr_started'):
    threading.Thread(target=worker, daemon=True).start()
    server = uvicorn.Server(uvicorn.Config(app, host='127.0.0.1', port=PORT, log_level='warning'))
    threading.Thread(target=server.run, daemon=True).start()
    _asr_started = True
for _ in range(30):
    try:
        print('ASR-Server bereit:', urllib.request.urlopen(f'http://127.0.0.1:{PORT}/health', timeout=5).read().decode())
        break
    except OSError:
        time.sleep(1)
else:
    raise RuntimeError('ASR-Server startet nicht')

## 5. HTTPS-Tunnel für den Orchestrator öffnen
Temporärer Cloudflare Quick Tunnel. Der Link ist öffentlich, Anfragen ohne Token werden abgewiesen. Die zwei `export`-Zeilen in `clipper/.env` eintragen (das `export` darf stehen bleiben).

In [ ]:
import os, re, subprocess, threading, time, urllib.request

CF = '/content/cloudflared'
if not os.path.exists(CF):
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', CF)
    os.chmod(CF, 0o755)
if globals().get('tunnel') is not None and tunnel.poll() is None:
    tunnel.terminate()
tunnel = subprocess.Popen([CF, 'tunnel', '--url', f'http://127.0.0.1:{PORT}', '--no-autoupdate'],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
tunnel_url = None
deadline = time.time() + 120
while tunnel_url is None and time.time() < deadline:
    line = tunnel.stdout.readline()
    if not line:
        break
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
if not tunnel_url:
    raise RuntimeError('Kein Tunnel-Link erkannt; Zelle erneut ausführen.')


def _drain_tunnel(proc=tunnel):
    # cloudflared blocks once its stdout pipe is full, so keep reading it.
    with open('/content/cloudflared.log', 'a') as log:
        for out in proc.stdout:
            log.write(out)


threading.Thread(target=_drain_tunnel, daemon=True).start()
print('In clipper/.env eintragen:\n')
print(f'export CLIPPER_ASR_URL={tunnel_url}')
print(f'export CLIPPER_ASR_TOKEN={API_TOKEN}')
print('\nWICHTIG: Ausgabe nicht teilen.')

## 6. Wach halten und Aufträge verfolgen
Hält die Sitzung beschäftigt und zeigt den Fortschritt laufender Aufträge. Mit ■ beenden; Server und Tunnel laufen weiter.

In [ ]:
last = None
try:
    while True:
        job_id = CURRENT['id']
        stage = JOBS[job_id].get('stage') if job_id else None
        if job_id and (job_id, stage) != last:
            print(time.strftime('%H:%M:%S'), job_id[:8], stage, flush=True)
        last = (job_id, stage)
        if tunnel.poll() is not None:
            print('Tunnel beendet - Abschnitt 5 erneut ausführen (neuer Link).')
            break
        time.sleep(5)
except KeyboardInterrupt:
    print('Beobachtung beendet; Server läuft weiter.')